In [ ]:
# %%
import os
import sys
import cv2
import json
import torch
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset

# Import des utilitaires YOLOP (Assure-toi que 'lib' est accessible)
sys.path.append(os.getcwd())
try:
    from lib.core.general import non_max_suppression, box_iou, ap_per_class
    from lib.core.evaluate import SegmentationMetric
except ImportError:
    print("❌ Erreur: Impossible d'importer les modules YOLOP (lib.core). Vérifie ton dossier.")

# Configuration
%matplotlib inline
ONNX_PATH = 'weights/yolop-640-640.onnx'
LIMIT_IMAGES = 100  # Nombre d'images à tester (None pour tout)
CONF_THRES = 0.001
IOU_THRES = 0.6

In [ ]:
# %%
class CenteredBDD(Dataset):
    def __init__(self, limit=100):
        # Adaptation des chemins à TON environnement
        self.img_root = 'dataset/images/val'
        self.label_root = 'dataset/det_annotations/val' # Attention: ici ce sont des JSON
        self.mask_root = 'dataset/da_seg_annotations/val'
        self.lane_root = 'dataset/ll_seg_annotations/val'
        
        self.img_list = sorted([os.path.join(self.img_root, f) for f in os.listdir(self.img_root) if f.endswith('.jpg')])
        if limit: 
            self.img_list = self.img_list[:limit]
            
    def __len__(self): 
        return len(self.img_list)

    def __getitem__(self, index):
        img_path = self.img_list[index]
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: 
            # Retour d'urgence si image illisible
            return torch.zeros((3,640,640)), torch.zeros((0,6)), img_path, torch.zeros(6)

        h_org, w_org = img_bgr.shape[:2]
        
        # --- LETTERBOX (Centré) ---
        img_size = 640
        input_img = np.full((img_size, img_size, 3), 114, dtype=np.uint8)
        r = min(img_size/h_org, img_size/w_org)
        new_w, new_h = int(w_org * r), int(h_org * r)
        img_resized = cv2.resize(img_bgr, (new_w, new_h))
        dw = (img_size - new_w) // 2
        dh = (img_size - new_h) // 2
        input_img[dh:dh+new_h, dw:dw+new_w] = img_resized
        
        # --- PREPROCESS ---
        input_img = input_img[:, :, ::-1] # BGR -> RGB
        img_norm = input_img.astype(np.float32) / 255.0
        img_norm = (img_norm - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
        img_norm = img_norm.transpose(2, 0, 1) # HWC -> CHW
        
        # --- LABELS (Chargement JSON) ---
        name = os.path.splitext(os.path.basename(img_path))[0]
        json_path = img_path.replace("images", "det_annotations").replace(".jpg", ".json")
        
        gt_boxes = []
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    frames = data.get('frames', [])
                    if frames:
                        objects = frames[0].get('objects', [])
                        for obj in objects:
                            # Filtre Véhicules Uniquement
                            if obj.get('category') in ['car', 'bus', 'truck', 'train']:
                                box = obj.get('box2d', {})
                                if box:
                                    # Format [class, x1, y1, x2, y2]
                                    gt_boxes.append([0, float(box['x1']), float(box['y1']), float(box['x2']), float(box['y2'])])
            except: pass
            
        targets = torch.tensor(gt_boxes, dtype=torch.float32) if len(gt_boxes) > 0 else torch.zeros((0, 5))
        
        # Infos de redimensionnement pour le post-process
        # [h, w, new_h, new_w, dh, dw]
        shape_info = torch.tensor([h_org, w_org, new_h, new_w, dh, dw], dtype=torch.float32)
        
        return torch.from_numpy(img_norm), targets, img_path, shape_info

    @staticmethod
    def collate_fn(batch):
        imgs, targets, paths, shapes = zip(*batch)
        # On doit gérer les targets de taille variable pour le batch
        # YOLOP attend une liste de targets où la première colonne est l'index de l'image dans le batch
        new_targets = []
        for i, t in enumerate(targets):
            if t.numel() > 0:
                # Ajoute l'index de l'image (i) comme première colonne
                # t est [n, 5] -> devient [n, 6] : [img_idx, class, x1, y1, x2, y2]
                img_idx = torch.full((t.shape[0], 1), i)
                new_targets.append(torch.cat((img_idx, t), 1))
        
        if new_targets:
            new_targets = torch.cat(new_targets, 0)
        else:
            new_targets = torch.zeros((0, 6))
            
        imgs = torch.stack(imgs, 0)
        shapes = torch.stack(shapes, 0)
        
        return imgs, new_targets, paths, shapes

In [ ]:
# %%
def eval_dataset(onnx_path, limit=100):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Dataset & Loader
    dataset = CenteredBDD(limit=limit)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2, collate_fn=CenteredBDD.collate_fn)
    
    print(f'🚀 Chargement ONNX : {onnx_path}')
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    try:
        session = onnxruntime.InferenceSession(onnx_path, providers=providers)
        input_name = session.get_inputs()[0].name
    except Exception as e:
        print(f"❌ Erreur ONNX : {e}")
        return

    # Metrics
    iouv = torch.linspace(0.5, 0.95, 10).to(device) # Vector IOU for mAP@0.5:0.95
    niou = iouv.numel()
    
    da_metric = SegmentationMetric(2)
    ll_metric = SegmentationMetric(2)
    
    # Stats détection : liste de tuples (correct, conf, pred_cls, target_cls)
    det_stats = []
    
    # Temps
    seen = 0
    t_inf = 0.0
    
    print(f'⏱️ Évaluation sur {len(dataset)} images...')
    
    # Boucle sur le dataset
    for i, (imgs, targets, paths, shapes) in tqdm(enumerate(loader), total=len(loader)):
        imgs = imgs.numpy()
        targets = targets.to(device)
        nb, _, height, width = imgs.shape
        
        # --- INFERENCE ---
        t_start = time.time()
        # ONNX Runtime (Batch=1 ici pour simplicité avec ORT Python)
        outs = session.run(None, {input_name: imgs})
        t_inf += time.time() - t_start
        
        # Sorties : [det, da_seg, ll_seg]
        det_out = torch.from_numpy(outs[0]).to(device)
        da_seg_out = torch.from_numpy(outs[1]).to(device)
        ll_seg_out = torch.from_numpy(outs[2]).to(device)
        
        # --- POST-PROCESS DETECTION ---
        # NMS
        out = non_max_suppression(det_out, conf_thres=CONF_THRES, iou_thres=IOU_THRES)
        
        # --- EVALUATION DETECTION ---
        for si, pred in enumerate(out):
            labels = targets[targets[:, 0] == si, 1:] # Récupère targets de l'image si
            nl = len(labels)
            tcls = labels[:, 0].tolist() if nl else []
            path = paths[si]
            
            # Infos de resize pour remettre à l'échelle
            h_org, w_org, new_h, new_w, dh, dw = shapes[si].tolist()
            r = min(640/h_org, 640/w_org)

            if len(pred) == 0:
                if nl:
                    det_stats.append((torch.zeros(0, niou, dtype=torch.bool), torch.Tensor(), torch.Tensor(), tcls))
                continue

            # Rescale boxes to original image
            pred[:, 0] -= dw
            pred[:, 1] -= dh
            pred[:, 2] -= dw
            pred[:, 3] -= dh
            pred[:, :4] /= r
            
            # Clip boxes
            pred[:, [0, 2]] = pred[:, [0, 2]].clamp(0, w_org)
            pred[:, [1, 3]] = pred[:, [1, 3]].clamp(0, h_org)

            # Assignation Detection (Metrics)
            correct = torch.zeros(pred.shape[0], niou, dtype=torch.bool, device=device)
            if nl:
                detected = []
                tcls_tensor = labels[:, 0]
                tbox = labels[:, 1:5]
                
                # Check per class
                for cls in torch.unique(tcls_tensor):
                    ti = (tcls_tensor == cls).nonzero(as_tuple=False).view(-1)
                    pi = (pred[:, 5] == cls).nonzero(as_tuple=False).view(-1)
                    
                    if pi.shape[0]:
                        # IoU Matrix [pred, target]
                        ious, i = box_iou(pred[pi, :4], tbox[ti]).max(1)
                        
                        # Check matches
                        for j in (ious > iouv[0]).nonzero(as_tuple=False):
                            d = ti[i[j]] # index target
                            if d.item() not in detected:
                                detected.append(d.item())
                                correct[pi[j]] = ious[j] > iouv
                                if len(detected) == nl: break
                                
            det_stats.append((correct.cpu(), pred[:, 4].cpu(), pred[:, 5].cpu(), tcls))

        # --- EVALUATION SEGMENTATION ---
        # DA
        _, da_predict = torch.max(da_seg_out, 1)
        da_predict = da_predict[:, int(dh):int(dh+new_h), int(dw):int(dw+new_w)]
        
        # LL
        _, ll_predict = torch.max(ll_seg_out, 1)
        ll_predict = ll_predict[:, int(dh):int(dh+new_h), int(dw):int(dw+new_w)]
        
        for si in range(nb):
            # Load GT
            name = os.path.splitext(os.path.basename(paths[si]))[0]
            da_gt_path = dataset.mask_root + '/' + name + '.png'
            ll_gt_path = dataset.lane_root + '/' + name + '.png'
            
            # Resize Pred to Original
            h_org, w_org = int(shapes[si][0]), int(shapes[si][1])
            
            # DA Metric
            if os.path.exists(da_gt_path):
                da_gt = cv2.imread(da_gt_path, 0)
                if da_gt is not None:
                    pred_mask = cv2.resize(da_predict[si].byte().cpu().numpy(), (w_org, h_org), interpolation=cv2.INTER_NEAREST)
                    da_gt = (da_gt > 0).astype(np.uint8) # Binary
                    da_metric.addBatch(pred_mask, da_gt)
            
            # LL Metric
            if os.path.exists(ll_gt_path):
                ll_gt = cv2.imread(ll_gt_path, 0)
                if ll_gt is not None:
                    pred_mask = cv2.resize(ll_predict[si].byte().cpu().numpy(), (w_org, h_org), interpolation=cv2.INTER_NEAREST)
                    ll_gt = (ll_gt > 0).astype(np.uint8)
                    # DILATATION (Astuce pour matcher les scores officiels)
                    kernel = np.ones((5, 5), np.uint8)
                    ll_gt = cv2.dilate(ll_gt, kernel, iterations=1)
                    
                    ll_metric.addBatch(pred_mask, ll_gt)
        
        seen += nb

    # --- CALCUL FINAL ---
    print(f"\n{'='*40}")
    print("📊 RÉSULTATS D'ÉVALUATION")
    print(f"{'='*40}")
    
    # 1. Speed
    t_avg = (t_inf / seen) * 1000
    fps = 1000 / t_avg
    print(f"Speed(fps) : {fps:.2f} ({t_avg:.2f} ms/img)")
    print("-" * 40)

    # 2. Detection (ap_per_class)
    if len(det_stats):
        stats = [np.concatenate(x, 0) for x in zip(*det_stats)]
        if len(stats) and stats[0].any():
            p, r, ap, f1, ap_class = ap_per_class(*stats, plot=False, save_dir=None, names={0: 'car'})
            ap50, ap = ap[:, 0], ap.mean(1)
            
            print("Traffic Object Detection Result")
            print(f"Recall(%)  : {r.mean()*100:.2f}")
            print(f"mAP50(%)   : {ap50.mean()*100:.2f}")
    print("-" * 40)

    # 3. Segmentation
    da_iou = da_metric.meanIntersectionOverUnion() * 100
    print("Drivable Area Segmentation Result")
    print(f"mIOU(%)    : {da_iou:.2f}")
    
    # 4. Lane Line
    # YOLOP officiel utilise souvent mIOU pour Lane Line (calculé comme moyenne IoU Fond + IoU Ligne)
    # L'IOU simple est l'IoU de la classe Ligne uniquement.
    ll_miou = ll_metric.meanIntersectionOverUnion() * 100 
    ll_iou = ll_metric.IntersectionOverUnion() * 100
    
    print("-" * 40)
    print("Lane Detection Result:")
    print(f"mIOU(%)    : {ll_miou:.2f}")
    print(f"IOU(%)     : {ll_iou:.2f}")
    print("=" * 40)

In [ ]:
# %%
if __name__ == "__main__":
    if os.path.exists(ONNX_PATH):
        eval_dataset(ONNX_PATH, limit=LIMIT_IMAGES)
    else:
        print(f"Erreur: Modèle introuvable {ONNX_PATH}")